# Category selectivity by Costas cell class — encoding vs delay (load 1, LME)

**Question.** Do exc and inh cells differ in how strongly they carry category information, during
encoding and during the delay?

**Model — one LME per epoch:**

```
resp ~ is_pref * C(celltype, Treatment(reference="exc"))

groups     = subject                       # subject random intercept
vc_formula = {'unit': '0 + C(unit_id)'}    # unit random intercept, NESTED within subject
fit(reml=True)
```

Random **intercepts only** (subject, and unit within subject). **No random slopes.**

The nesting is structural: statsmodels builds the variance component *inside* each `groups` level, so
`unit` is a unit-within-subject intercept with one shared variance parameter. It is only truly nested
if each `unit_id` belongs to one subject — `unit_id` carries the subject prefix
(`sub-N_ses-M_..._uNNN`), and `[1.1]` asserts it.

- `resp`     per-trial rate, baseline z-scored per cell: `(rate − μ)/σ`, μ σ from the per-trial rates
             in the **−0.9 to −0.3 s** window before enc1 onset (inside fixation, starts −1.084 s)
- `is_pref`  1 when the encoded picture was this cell's preferred category
- `celltype` Costas `costas_class`, exc / inh, treatment-coded with **exc** as reference

From each single fit we read the `is_pref` **β for both classes** (exc directly; inh = exc +
interaction, with a delta-method SE so it is reference-invariant), each class's **β-vs-0** Wald p, and
the **`is_pref × celltype` interaction** — the inh−exc difference in that epoch, and its p.

**Epochs, fit separately:** encoding = 0.2–1.0 s re enc1 onset; delay = 0–2.5 s re maintenance onset.

**Design.** Load-1 trials only, so the held item is unambiguously the enc1 picture. Cells = the 271
category cells from `category_cells_<region>.csv`. Class = Costas only.

**QC.** `MIN_CLASS_CONF` (default 0.70) drops cells whose exc/inh call is near the 0.50 chance level
for a 2-way choice. It cuts **both** arms of the contrast — unlike Section 3's subtype threshold,
which holds exc whole — and inh is the harder class to call, so `[1.1]` prints the resulting class
balance as well as the counts. Section 3 stacks its subtype threshold on top of this.

**Timing** (load-1 medians, n = 3073), locked to enc1 onset: fixation −1.084 s, **enc1 onset 0**,
enc1 off / maintenance on **+2.016 s**, probe **+4.701 s**.

**Interpretation limit.** Selection used encoding + probe, so the **encoding** epoch is circular while
the **delay** was held out and is clean. The encoding β is inflated for both classes; read the
within-epoch class contrast, not the encoding-to-delay change.

000673, hippocampus + amygdala.

## Section 0 — Setup

In [ ]:
# [0.1] imports, config
import sys, warnings, time, json
from pathlib import Path
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
import statsmodels.formula.api as smf

PROJECT = Path.cwd()                        # run from E:\SBCAT\celltyping
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from celltyping.selectivity import load_spikes_trials, trial_table, window_counts

RS = 42
plt.rcParams['figure.dpi'] = 100

# ---- CONFIG ----
DATASET      = '000673'
DATA_ROOT    = PROJECT.parent / 'data' / DATASET
CT           = PROJECT / 'outputs' / 'celltype'
SELECT_AREAS = ['hippocampus', 'amygdala']
region_tag   = '+'.join(a.lower() for a in SELECT_AREAS)
CAT_CELLS_CSV = CT / f'category_cells_{region_tag}.csv'
COSTAS_CSV    = CT / f'unit_labels_costas_{region_tag}.csv'

LOAD       = 1                     # load-1 only: held item == enc1 picture
BASELINE   = (-0.9, -0.3)          # re enc1 onset, inside fixation
SD_FLOOR   = 0.1                   # Hz; floor on baseline SD
ENC_WIN    = (0.2, 1.0)            # re enc1 onset
DELAY_WIN  = (0.0, 2.5)            # re maintenance onset
EPOCHS     = ['encoding', 'delay']
MIN_TRIALS = 5                     # per cell, in EACH of preferred / non-preferred
REF_CLASS  = 'exc'                 # treatment-coding reference

# QC floor on the exc/inh call (costas_class_conf). Chance for a 2-way call is 0.50, and the bottom
# ~10% of cells sit below 0.64, so this drops the near-coin-flip tail. Applied in [1.1], and Section
# 3's subtype threshold stacks ON TOP of it. Set to None to keep every cell.
MIN_CLASS_CONF = 0.70

PSTH_WIN = (-1.0, 4.7); BIN_MS = 50.0; SIGMA_MS = 50.0
T_ENC_ON, T_MAINT_ON, T_PROBE_ON = 0.0, 2.016, 4.701

GROUPS  = ['inh', 'exc']
GRP_COL = {'inh': '#d62728', 'exc': '#1f77b4'}
ALPHA   = 0.05
print(f'{DATASET} | {SELECT_AREAS} | load-{LOAD} | baseline {BASELINE}s re enc1 | LME per epoch')

In [ ]:
# [0.2] load spikes + trials, join the category-cell selection and the Costas class
t0 = time.time()
files = sorted(DATA_ROOT.glob('sub-*/*.nwb'))
assert files, f'no NWB files under {DATA_ROOT}'
units_df, spikes, trials_by_session = load_spikes_trials(files, areas=SELECT_AREAS, verbose=False)
units_df = units_df.reset_index(drop=True)
uid2idx = {u: i for i, u in enumerate(units_df.unit_id)}

sel = pd.read_csv(CAT_CELLS_CSV)
cos = pd.read_csv(COSTAS_CSV)[['unit_id', 'costas_class']].rename(columns={'costas_class': 'celltype'})
cc = sel[sel.selective].merge(cos, on='unit_id', how='left')
cc = cc[cc.celltype.notna()].copy()
print(f'{len(units_df)} units in {time.time()-t0:.0f}s | {len(cc)} category cells with a Costas class')
print('  by class:', cc.celltype.value_counts().to_dict())

## Section 1 — Trial-level long table

**One row per cell × trial × epoch** — what the LME is fit on. `resp` is that trial's rate in the
epoch window, baseline z-scored with the cell's own μ, σ from the fixation window. `epoch` is only a
subsetting key here; each epoch gets its own model.

`at_floor` flags cells whose baseline SD fell below `SD_FLOOR`, so `resp` is scaled by the floor
constant rather than the cell's own variability. Reported per class.

In [ ]:
# [1.1] build the trial-level long table (unit x trial x epoch)
# ---- QC: class-confidence floor on the exc/inh call ----
# Applied here rather than in [0.2] so MIN_CLASS_CONF can be retuned without reloading the NWB files.
# Unlike Section 3's subtype threshold, this one cuts BOTH arms of the contrast (it is the confidence
# in the exc/inh call itself), and inh is the harder class to call - so the class balance shifts too.
# Both are printed. Section 3 inherits this filter: d5 is built from long_df.
_conf = pd.read_csv(COSTAS_CSV).set_index('unit_id').costas_class_conf
cc_qc = cc.copy()
if MIN_CLASS_CONF is not None:
    _before = cc_qc.groupby('celltype').size().to_dict()
    _f_before = 100 * (cc_qc.celltype == 'inh').mean()
    cc_qc = cc_qc[cc_qc.unit_id.map(_conf) >= MIN_CLASS_CONF].copy()
    print(f'class-conf QC at {MIN_CLASS_CONF}: {_before} -> {cc_qc.groupby("celltype").size().to_dict()}'
          f'   ({len(cc) - len(cc_qc)} of {len(cc)} cells dropped)')
    print(f'  inh fraction: {_f_before:.1f}% -> {100 * (cc_qc.celltype == "inh").mean():.1f}%')
else:
    print('class-conf QC: OFF (MIN_CLASS_CONF is None)')

rows, cellinfo = [], []
for _, r in cc_qc.iterrows():
    j = uid2idx.get(r.unit_id)
    if j is None or pd.isna(r.pref_cat):
        continue
    spk = spikes[j]
    T = trial_table(trials_by_session.get(r.session_id))
    e1 = T.enc1_on.to_numpy(float); mo = T.maint_on.to_numpy(float)
    c1 = T.enc1_cat.to_numpy(float); ld = T.loads.to_numpy(float)
    ok = (ld == LOAD) & np.isfinite(e1) & (e1 > 0) & np.isfinite(c1)
    if ok.sum() < 2 * MIN_TRIALS:
        continue
    br = window_counts(spk, e1[ok], *BASELINE) / (BASELINE[1] - BASELINE[0])
    mu = float(br.mean()); sd_raw = float(br.std()); sd = max(sd_raw, SD_FLOOR)
    ip_all = (c1[ok] == r.pref_cat)
    if ip_all.sum() < MIN_TRIALS or (~ip_all).sum() < MIN_TRIALS:
        continue
    tid = np.arange(int(ok.sum()))
    for epoch, onsets, win in (('encoding', e1[ok], ENC_WIN), ('delay', mo[ok], DELAY_WIN)):
        g = np.isfinite(onsets) & (onsets > 0)
        if (ip_all & g).sum() < MIN_TRIALS or ((~ip_all) & g).sum() < MIN_TRIALS:
            continue
        z = (window_counts(spk, onsets[g], *win) / (win[1] - win[0]) - mu) / sd
        for zz, ii, tt in zip(z, ip_all[g].astype(int), tid[g]):
            rows.append((r.unit_id, r.subject, r.area, r.celltype, epoch, int(tt), int(ii), float(zz)))
    cellinfo.append(dict(unit_id=r.unit_id, subject=r.subject, celltype=r.celltype,
                         base_mean=mu, base_sd=sd_raw, at_floor=bool(sd_raw < SD_FLOOR)))

long_df = pd.DataFrame(rows, columns=['unit_id', 'subject', 'area', 'celltype',
                                      'epoch', 'trial', 'is_pref', 'resp'])
CI = pd.DataFrame(cellinfo)

# NESTING GUARD: the unit variance component is only nested inside subject if each unit_id belongs
# to exactly one subject. If a unit_id spanned two subjects the vc would be CROSSED, not nested.
_chk = long_df.groupby('unit_id').subject.nunique()
assert int(_chk.max()) == 1, f'{int((_chk > 1).sum())} unit_id(s) span >1 subject — vc not nested'
print(f'nesting OK: all {len(_chk)} units belong to exactly one subject '
      f'(unit within subject, {long_df.subject.nunique()} subjects)')

print(f'long table: {long_df.shape[0]:,} rows  |  {long_df.unit_id.nunique()} cells  |  '
      f'{long_df.subject.nunique()} subjects')
print(long_df.groupby(['epoch', 'celltype']).size().unstack().to_string())
print(f'\nbaseline SD floor ({SD_FLOOR} Hz):')
print(CI.groupby('celltype').agg(n=('unit_id', 'size'),
      pct_at_floor=('at_floor', lambda s: 100 * s.mean()),
      median_base_sd=('base_sd', 'median')).round(2).to_string())

## Section 2 — The LME, one fit per epoch

`resp ~ is_pref * C(celltype, Treatment(reference="exc"))`, `groups = subject`,
`vc_formula = {'unit': '0 + C(unit_id)'}`, `reml=True`. Random intercepts only.

Both class βs come from the **same** fit, so they are jointly estimated and directly comparable
within that epoch, and the interaction term *is* the inh−exc difference.

`[2.3]` is the descriptive PSTH for the same cells — one continuous trace per class locked to enc1
onset, spanning fixation → enc1 → maintenance.

In [ ]:
# [2.1] one LME per epoch (subject + unit random INTERCEPTS, no slopes)
from scipy.stats import norm, chi2

def _fit(sub, formula):
    # NESTED, unit within subject. `groups=subject` is the subject random intercept; the variance
    # component is constructed INSIDE each subject group, so 'unit' is a unit-within-subject random
    # intercept sharing one variance parameter across subjects. unit_id carries the subject prefix
    # (sub-N_ses-M_..._uNNN), so every unit belongs to exactly one subject -> genuinely nested and
    # not crossed. That precondition is asserted in [1.1].
    md = smf.mixedlm(formula, sub, groups=sub['subject'], vc_formula={'unit': '0 + C(unit_id)'})
    return md.fit(reml=True)

def _interaction_betas(mdf, pred, groups, ref):
    # Read both class betas off ONE treatment-coded fit. `ref` is the pred term directly; every other
    # group is ref + its interaction, with a delta-method SE -> reference-INVARIANT. Also returns
    # each group's beta-vs-0 Wald p, each group's difference-from-ref p (the interaction term), and
    # the joint Wald that all interaction terms are 0 (for 2 groups this equals the single p).
    p = mdf.params; V = mdf.cov_params(); pv = mdf.pvalues
    beta = {ref: float(p[pred])}; se = {ref: float(mdf.bse[pred])}; p0 = {ref: float(pv[pred])}
    vsref, it_terms = {}, []
    for g in groups:
        if g == ref:
            continue
        cand = [t for t in p.index if t.startswith(pred + ':') and (f'[T.{g}]' in t)]
        if not cand:
            continue
        it = cand[0]; it_terms.append(it)
        beta[g] = float(p[pred] + p[it])
        se[g] = float(np.sqrt(V.loc[pred, pred] + V.loc[it, it] + 2 * V.loc[pred, it]))
        p0[g] = float(2 * (1 - norm.cdf(abs(beta[g] / se[g]))))
        vsref[g] = float(pv[it])
    omni = np.nan
    if it_terms:
        b = p[it_terms].to_numpy(float); S = V.loc[it_terms, it_terms].to_numpy(float)
        try:
            omni = float(chi2.sf(float(b @ np.linalg.solve(S, b)), len(it_terms)))
        except Exception:
            omni = np.nan
    return beta, se, p0, vsref, omni

FORMULA = f'resp ~ is_pref * C(celltype, Treatment(reference="{REF_CLASS}"))'
print(FORMULA + "   |   groups=subject, vc={'unit': '0 + C(unit_id)'}\n")

rows, diff_p, fits = [], {}, {}
for epoch in EPOCHS:
    sub = long_df[long_df.epoch == epoch].copy()
    if sub.celltype.nunique() < 2:
        continue
    t0 = time.time()
    mdf = _fit(sub, FORMULA)
    fits[epoch] = mdf
    beta, se, p0, vsref, omni = _interaction_betas(mdf, 'is_pref', GROUPS, REF_CLASS)
    diff_p[epoch] = omni
    print(f'{epoch:9s}: {len(sub):,} rows, {sub.unit_id.nunique()} cells, '
          f'converged={mdf.converged}, {time.time()-t0:.0f}s')
    for g in GROUPS:
        if g in beta:
            rows.append(dict(epoch=epoch, celltype=g,
                             n_cells=int(sub[sub.celltype == g].unit_id.nunique()),
                             beta=beta[g], se=se[g], p_vs0=p0[g]))
beta_df = pd.DataFrame(rows)

print('\nis_pref beta per epoch x class (both classes from the SAME fit):')
print(beta_df.round(4).to_string(index=False))
print('\ninh-vs-exc difference (is_pref x celltype interaction), per epoch:')
for epoch in EPOCHS:
    print(f'  {epoch:9s}: p = {diff_p.get(epoch, float("nan")):.4g}')

In [ ]:
# [2.2] beta per epoch per class, +-1 SE; vs-0 stars; class-difference p annotated per epoch
def _stars(pv):
    if not np.isfinite(pv):
        return ''
    return '***' if pv < 1e-3 else ('**' if pv < 1e-2 else ('*' if pv < 0.05 else ''))

def _star_bar(ax, x, beta, se, pv, fs=11):
    s = _stars(pv)
    if s:
        top = beta + se if beta >= 0 else beta - se
        ax.text(x, top, s, ha='center', va='bottom' if beta >= 0 else 'top', fontsize=fs)

fig, ax = plt.subplots(figsize=(6.4, 4.2))
xw = np.arange(len(EPOCHS))
for g in GROUPS:
    b = beta_df[beta_df.celltype == g].set_index('epoch').reindex(EPOCHS)
    ax.errorbar(xw, b.beta, yerr=b.se, color=GRP_COL[g], marker='o', ms=7, lw=1.8, capsize=3,
                label=f'{g} (n={int(b.n_cells.dropna().max()) if b.n_cells.notna().any() else 0})')
    for x, bb in zip(xw, b.itertuples()):
        _star_bar(ax, x, bb.beta, bb.se, bb.p_vs0)
ylim = ax.get_ylim()[1]
for i, epoch in enumerate(EPOCHS):
    if np.isfinite(diff_p.get(epoch, np.nan)):
        ax.text(i, ylim * 0.94, f'inh vs exc\np={diff_p[epoch]:.2g}', ha='center', fontsize=8)
ax.axhline(0, color='k', lw=0.7)
ax.set_xticks(xw); ax.set_xticklabels(EPOCHS); ax.set_xlim(-0.35, len(EPOCHS) - 0.65)
ax.set_ylabel('selectivity β ± 1 SE  (is_pref, baseline-z units)')
ax.legend(title='Costas')
ax.set_title('Category selectivity β by epoch and cell class\n(one LME per epoch, load-1)', fontsize=10)
fig.tight_layout(); plt.show()

In [ ]:
# [2.3] PSTH: one continuous trace per class, fixation -> enc1 -> maintenance, locked to enc1 onset
bin_s = BIN_MS / 1000.0
edges = np.arange(PSTH_WIN[0], PSTH_WIN[1] + bin_s, bin_s)
tc = (edges[:-1] + edges[1:]) / 2.0

def _traces(spk, onsets):
    st = np.sort(np.asarray(spk, float))
    M = np.empty((len(onsets), edges.size - 1))
    for i, o in enumerate(onsets):
        lo = np.searchsorted(st, o + PSTH_WIN[0], 'left'); hi = np.searchsorted(st, o + PSTH_WIN[1], 'right')
        c = np.histogram(st[lo:hi] - o, bins=edges)[0] / bin_s
        M[i] = gaussian_filter1d(c, SIGMA_MS / BIN_MS) if SIGMA_MS > 0 else c
    return M

keep = set(long_df.unit_id)
acc = {g: {'pref': [], 'non': [], 'cells': 0} for g in GROUPS}
for _, r in cc.iterrows():
    if r.unit_id not in keep:
        continue
    j = uid2idx.get(r.unit_id)
    T = trial_table(trials_by_session.get(r.session_id))
    e1 = T.enc1_on.to_numpy(float); c1 = T.enc1_cat.to_numpy(float); ld = T.loads.to_numpy(float)
    ok = (ld == LOAD) & np.isfinite(e1) & (e1 > 0) & np.isfinite(c1)
    if ok.sum() < 2 * MIN_TRIALS:
        continue
    M = _traces(spikes[j], e1[ok])
    # matched-resolution baseline: same 50 ms bins + smoothing as the trace, so the height is honest
    bb = M[:, (tc >= BASELINE[0]) & (tc <= BASELINE[1])]
    Z = (M - float(bb.mean())) / max(float(bb.std()), SD_FLOOR)
    ip = (c1[ok] == r.pref_cat)
    if ip.sum() < MIN_TRIALS or (~ip).sum() < MIN_TRIALS:
        continue
    acc[r.celltype]['pref'].append(Z[ip]); acc[r.celltype]['non'].append(Z[~ip])
    acc[r.celltype]['cells'] += 1

fig, axes = plt.subplots(len(GROUPS), 1, figsize=(9, 3.1 * len(GROUPS)), sharex=True, sharey=True)
for ax, g in zip(np.atleast_1d(axes), GROUPS):
    for key, col_, ls, name in (('pref', GRP_COL[g], '-', 'preferred category'),
                                ('non', '#888', '--', 'other categories')):
        A = acc[g][key]
        if not A:
            continue
        A = np.vstack(A); m_ = A.mean(0); s_ = A.std(0) / np.sqrt(len(A))
        ax.plot(tc, m_, color=col_, ls=ls, lw=2, label=f'{name}  ({A.shape[0]} trials)')
        ax.fill_between(tc, m_ - s_, m_ + s_, color=col_, alpha=0.15)
    ax.axvspan(PSTH_WIN[0], T_ENC_ON, color='#999', alpha=0.10)
    ax.axvspan(T_ENC_ON, T_MAINT_ON, color='#f0c419', alpha=0.13)
    ax.axvspan(T_MAINT_ON, T_PROBE_ON, color='#4c9be8', alpha=0.10)
    for xv in (T_ENC_ON, T_MAINT_ON, T_PROBE_ON):
        ax.axvline(xv, color='k', lw=0.8, ls=':')
    ax.axhline(0, color='k', lw=0.6, alpha=0.5)
    ax.set_ylabel(f'{g}  (n={acc[g]["cells"]} cells)\nz to baseline')
    ax.legend(fontsize=8, loc='upper right')
ax.set_xlabel('time from enc1 onset (s)'); ax.set_xlim(*PSTH_WIN)
np.atleast_1d(axes)[0].set_title('fixation  |  enc1 on screen  |  maintenance      '
                                 '(load-1, locked to enc1 onset)', fontsize=10)
fig.tight_layout(); plt.show()

## Section 3 — Breaking down the inhibitory types  (exploratory)

Same model, same random effects, **5 groups instead of 2**: `exc` kept whole as the reference, and the
inhibitory cells split into their Costas subtypes **PVALB / SST / VIP / LAMP5**.

```
resp ~ is_pref * C(grp, Treatment(reference="exc"))
groups = subject,  vc_formula = {'unit': '0 + C(unit_id)'},  reml=True
```

One fit per epoch, exactly as in `[2.1]` — the only change is that the class factor has 5 levels
instead of 2. From that single fit, `_interaction_betas` returns each group's `is_pref` β (exc read
directly; each subtype = exc + its interaction, delta-method SE, so reference-invariant), each group's
β-vs-0 Wald p, each **subtype-vs-exc** p (the interaction terms), and the **omnibus** — a joint Wald
that all four interaction terms are zero, i.e. the >2-group generalisation of `[2.1]`'s single
interaction p.

`[3.4]` is the matching PSTH — same recipe as `[2.3]`, one panel per group.

`[3.1]` reports cells per subtype and the Costas confidence for both the class and the subtype call,
each against its chance level (class is a 2-way choice, chance 0.50; subtype is a 4-way choice within
inh, chance 0.25 — `costas_type` nests strictly inside `costas_class`).

In [ ]:
# [3.1] attach Costas subtype, build the 5-group factor (exc whole + 4 inh subtypes)
# Self-contained: merges onto long_df by unit_id, so nothing in Sections 0-2 needs re-running.
INH_SUBTYPES = ['PVALB', 'SST', 'VIP', 'LAMP5']
GROUPS5 = ['exc'] + INH_SUBTYPES
GC5 = {'exc': '#1f77b4', 'PVALB': '#8c564b', 'SST': '#d62728', 'VIP': '#2ca02c', 'LAMP5': '#ff7f0e'}

_sub = pd.read_csv(COSTAS_CSV)[['unit_id', 'costas_type', 'costas_type_conf']]
d5 = long_df.merge(_sub, on='unit_id', how='left')
d5['grp'] = np.where(d5.celltype == 'exc', 'exc',
                     np.where(d5.costas_type.isin(INH_SUBTYPES), d5.costas_type, None))
dropped = d5[d5.grp.isna()].unit_id.nunique()
d5 = d5[d5.grp.notna()].copy()

cells5 = d5.drop_duplicates('unit_id')
tab5 = (cells5.groupby('grp').agg(n_cells=('unit_id', 'size'),
                                  median_type_conf=('costas_type_conf', 'median'))
        .reindex(GROUPS5))
print('cells per group (exc kept whole, inh split by Costas subtype):')
print(tab5.round(2).to_string())
if dropped:
    print(f'\n{dropped} inh cell(s) dropped - costas_type not one of {INH_SUBTYPES}')

# Confidence vs CHANCE. costas_type nests strictly inside costas_class (ITL* only ever exc,
# LAMP5/PVALB/SST/VIP only ever inh), so the subtype call is a 4-way choice WITHIN inh -> chance
# 0.25, not 1/7. Class is a 2-way choice -> chance 0.50.
_all = pd.read_csv(COSTAS_CSV)
_inh_types = sorted(_all.loc[_all.costas_class == 'inh', 'costas_type'].dropna().unique())
_ch_type, _ch_class = 1.0 / max(len(_inh_types), 1), 0.5
print(f'\nconfidence vs chance:')
print(f'  CLASS exc/inh : median {_all.costas_class_conf.median():.2f}   '
      f'2-way -> chance {_ch_class:.2f}   ({_all.costas_class_conf.median()/_ch_class:.1f}x)')
print(f'  TYPE  in inh  : median {cells5.costas_type_conf.median():.2f}   '
      f'{len(_inh_types)}-way -> chance {_ch_type:.2f}   '
      f'({cells5.costas_type_conf.median()/_ch_type:.1f}x)')

In [ ]:
# [3.2] same LME per epoch, 5-level group factor (exc = reference)
FORMULA5 = 'resp ~ is_pref * C(grp, Treatment(reference="exc"))'
print(FORMULA5 + "   |   groups=subject, vc={'unit': '0 + C(unit_id)'}\n")

rows5, omni5, vsexc5, fits5 = [], {}, {}, {}
for epoch in EPOCHS:
    sub = d5[d5.epoch == epoch].copy()
    if sub.grp.nunique() < 2:
        continue
    t0 = time.time()
    mdf = _fit(sub, FORMULA5)
    fits5[epoch] = mdf
    beta, se, p0, vsref, omni = _interaction_betas(mdf, 'is_pref', GROUPS5, 'exc')
    omni5[epoch] = omni
    print(f'{epoch:9s}: {len(sub):,} rows, {sub.unit_id.nunique()} cells, '
          f'converged={mdf.converged}, {time.time()-t0:.0f}s')
    for g in GROUPS5:
        if g in beta:
            rows5.append(dict(epoch=epoch, grp=g,
                              n_cells=int(sub[sub.grp == g].unit_id.nunique()),
                              beta=beta[g], se=se[g], p_vs0=p0[g]))
        vsexc5[(epoch, g)] = vsref.get(g, np.nan)
bt5 = pd.DataFrame(rows5)

print('\nis_pref beta per epoch x group (one LME per epoch, exc = reference):')
print(bt5.round(4).to_string(index=False))
print('\ngroup x is_pref OMNIBUS (all 4 subtypes vs exc jointly), per epoch:')
for epoch in EPOCHS:
    print(f'  {epoch:9s}: p = {omni5.get(epoch, float("nan")):.4g}')
print('\nsubtype-vs-exc difference, per epoch:')
for epoch in EPOCHS:
    print(f'  {epoch:9s}: ' + '   '.join(
        f'{g}: {vsexc5.get((epoch, g), float("nan")):.3g}' for g in INH_SUBTYPES))

In [ ]:
# [3.3] beta per epoch per group, +-1 SE; vs-0 stars; omnibus p annotated per epoch
fig, ax = plt.subplots(figsize=(7.6, 4.4))
xw = np.arange(len(EPOCHS))
for g in GROUPS5:
    b = bt5[bt5.grp == g].set_index('epoch').reindex(EPOCHS)
    n = int(b.n_cells.dropna().max()) if b.n_cells.notna().any() else 0
    ax.errorbar(xw, b.beta, yerr=b.se, color=GC5[g], marker='o', ms=6, lw=1.6, capsize=3,
                label=f'{g} (n={n})')
    for x, bb in zip(xw, b.itertuples()):
        _star_bar(ax, x, bb.beta, bb.se, bb.p_vs0, fs=9)
ylim = ax.get_ylim()[1]
for i, epoch in enumerate(EPOCHS):
    if np.isfinite(omni5.get(epoch, np.nan)):
        ax.text(i, ylim * 0.96, f'omnibus\np={omni5[epoch]:.2g}', ha='center', fontsize=8)
ax.axhline(0, color='k', lw=0.7)
ax.set_xticks(xw); ax.set_xticklabels(EPOCHS); ax.set_xlim(-0.35, len(EPOCHS) - 0.65)
ax.set_ylabel('selectivity β ± 1 SE  (is_pref, baseline-z units)')
ax.set_title('Category selectivity β by epoch: exc vs inhibitory subtypes  [exploratory]', fontsize=10)
ax.legend(title='Costas', fontsize=8, ncol=5, loc='upper center', bbox_to_anchor=(0.5, -0.12))
fig.tight_layout(); plt.show()

In [ ]:
# [3.4] PSTH per group: exc + the 4 inh subtypes, locked to enc1 onset (same recipe as [2.3])
grp_of = dict(zip(cells5.unit_id, cells5.grp))
acc5 = {g: {'pref': [], 'non': [], 'cells': 0} for g in GROUPS5}
for _, r in cc.iterrows():
    g = grp_of.get(r.unit_id)
    if g is None:
        continue
    j = uid2idx.get(r.unit_id)
    T = trial_table(trials_by_session.get(r.session_id))
    e1 = T.enc1_on.to_numpy(float); c1 = T.enc1_cat.to_numpy(float); ld = T.loads.to_numpy(float)
    ok = (ld == LOAD) & np.isfinite(e1) & (e1 > 0) & np.isfinite(c1)
    if ok.sum() < 2 * MIN_TRIALS:
        continue
    M = _traces(spikes[j], e1[ok])
    bb = M[:, (tc >= BASELINE[0]) & (tc <= BASELINE[1])]      # matched-resolution baseline
    Z = (M - float(bb.mean())) / max(float(bb.std()), SD_FLOOR)
    ip = (c1[ok] == r.pref_cat)
    if ip.sum() < MIN_TRIALS or (~ip).sum() < MIN_TRIALS:
        continue
    acc5[g]['pref'].append(Z[ip]); acc5[g]['non'].append(Z[~ip]); acc5[g]['cells'] += 1

fig, axes = plt.subplots(len(GROUPS5), 1, figsize=(9, 2.4 * len(GROUPS5)), sharex=True, sharey=True)
for ax, g in zip(np.atleast_1d(axes), GROUPS5):
    for key, col_, ls, name in (('pref', GC5[g], '-', 'preferred category'),
                                ('non', '#888', '--', 'other categories')):
        A = acc5[g][key]
        if not A:
            continue
        A = np.vstack(A); m_ = A.mean(0); s_ = A.std(0) / np.sqrt(len(A))
        ax.plot(tc, m_, color=col_, ls=ls, lw=1.8, label=f'{name}  ({A.shape[0]} tr)')
        ax.fill_between(tc, m_ - s_, m_ + s_, color=col_, alpha=0.15)
    ax.axvspan(PSTH_WIN[0], T_ENC_ON, color='#999', alpha=0.10)
    ax.axvspan(T_ENC_ON, T_MAINT_ON, color='#f0c419', alpha=0.13)
    ax.axvspan(T_MAINT_ON, T_PROBE_ON, color='#4c9be8', alpha=0.10)
    for xv in (T_ENC_ON, T_MAINT_ON, T_PROBE_ON):
        ax.axvline(xv, color='k', lw=0.8, ls=':')
    ax.axhline(0, color='k', lw=0.6, alpha=0.5)
    ax.set_ylabel(f'{g}\n(n={acc5[g]["cells"]} cells)', fontsize=9)
    ax.legend(fontsize=6, loc='upper right')
ax.set_xlabel('time from enc1 onset (s)'); ax.set_xlim(*PSTH_WIN)
np.atleast_1d(axes)[0].set_title('fixation  |  enc1 on screen  |  maintenance      '
                                 'exc vs inhibitory subtypes  [exploratory]', fontsize=10)
fig.tight_layout(); plt.show()

### Confidence titration

Refit `[3.2]`'s model across a grid of thresholds θ on `costas_type_conf`, keeping inh cells with
conf ≥ θ. **exc is never thresholded** — it is not being subtyped here, so cutting it too would
confound "fewer inh cells" with "a different exc reference". The threshold uses the **subtype**
confidence only.

Every θ discards cells, so β moves and SE grows whether or not confidence carries information. The
**matched-n random-drop control** separates those: at each θ, keep the same number of cells *per
subtype* but chosen at random rather than by confidence, refit, repeat. That gives the spread of β
attributable to sample size alone. A confidence-thresholded β outside that band is doing something
random dropping does not.

The θ's are **nested subsets**, so their p-values are strongly correlated. This is a stability
display, not a family of tests: no correction across θ, and the primary result stays θ = the lowest
grid point (all cells).

In [ ]:
# [3.5] confidence titration — refit the Section 3 model across thresholds on costas_type_conf
CONF_GRID  = [0.30, 0.40, 0.45, 0.50, 0.55, 0.60]   # subtype-confidence thresholds
MIN_CELLS  = 5                                       # skip a theta if any subtype falls below this
RANDOM_DROP_CONTROL = True                           # matched-n control (set False to skip)
N_DROP_REPEATS = 20

rows_t, omni_t = [], []
for th in CONF_GRID:
    sub_all = d5[(d5.grp == 'exc') | (d5.costas_type_conf >= th)]     # exc never thresholded
    ncell = (sub_all.drop_duplicates('unit_id').groupby('grp').size()
             .reindex(GROUPS5).fillna(0).astype(int))
    if (ncell[INH_SUBTYPES] < MIN_CELLS).any():
        print(f'theta={th:.2f}  SKIPPED (min subtype n={int(ncell[INH_SUBTYPES].min())} '
              f'< {MIN_CELLS})   n={ncell.to_dict()}')
        continue
    print(f'theta={th:.2f}  n={ncell.to_dict()}')
    for epoch in EPOCHS:
        s = sub_all[sub_all.epoch == epoch]
        if s.grp.nunique() < 2:
            continue
        mdf = _fit(s, FORMULA5)
        beta, se, p0, vsref, omni = _interaction_betas(mdf, 'is_pref', GROUPS5, 'exc')
        omni_t.append(dict(theta=th, epoch=epoch, omnibus_p=omni,
                           n_inh=int(ncell[INH_SUBTYPES].sum()), converged=bool(mdf.converged)))
        for g in GROUPS5:
            if g in beta:
                rows_t.append(dict(theta=th, epoch=epoch, grp=g, n_cells=int(ncell[g]),
                                   beta=beta[g], se=se[g], p_vs0=p0[g],
                                   p_vs_exc=vsref.get(g, np.nan)))
tit = pd.DataFrame(rows_t); tit_omni = pd.DataFrame(omni_t)

print('\nomnibus (all subtypes vs exc jointly) across thresholds:')
print(tit_omni.pivot_table(index='theta', columns='epoch', values='omnibus_p').round(4).to_string())
print('\nper-group beta across thresholds:')
print(tit.pivot_table(index=['epoch', 'grp'], columns='theta', values='beta').round(3).to_string())
print('\nper-group SE across thresholds:')
print(tit.pivot_table(index=['epoch', 'grp'], columns='theta', values='se').round(3).to_string())
print('\nsubtype-vs-exc p across thresholds (the interaction terms; uncorrected):')
print(tit[tit.grp != 'exc'].pivot_table(index=['epoch', 'grp'], columns='theta',
                                        values='p_vs_exc').round(4).to_string())
print('\nbeta-vs-0 p across thresholds:')
print(tit.pivot_table(index=['epoch', 'grp'], columns='theta', values='p_vs0').round(4).to_string())

for _th in sorted(tit.theta.unique()):
    print(f'\n--- full table at theta = {_th:.2f} ---')
    print(tit[tit.theta == _th][['epoch', 'grp', 'n_cells', 'beta', 'se', 'p_vs0', 'p_vs_exc']]
          .round(4).to_string(index=False))

In [ ]:
# [3.6] matched-n random-drop control: same n per subtype, cells chosen at RANDOM not by confidence
ctrl_df = pd.DataFrame()
if RANDOM_DROP_CONTROL and len(tit):
    rng = np.random.default_rng(RS)
    inh_units = d5[d5.grp != 'exc'].drop_duplicates('unit_id')[['unit_id', 'grp']]
    ctrl, t0 = [], time.time()
    for th in sorted(tit.theta.unique()):
        want = (tit[(tit.theta == th) & (tit.grp != 'exc')]
                .drop_duplicates('grp').set_index('grp').n_cells)
        for rep in range(N_DROP_REPEATS):
            keep = []
            for g, n in want.items():
                pool = inh_units.loc[inh_units.grp == g, 'unit_id'].to_numpy()
                keep.extend(rng.choice(pool, size=min(int(n), pool.size), replace=False))
            s_all = d5[(d5.grp == 'exc') | d5.unit_id.isin(keep)]
            for epoch in EPOCHS:
                s = s_all[s_all.epoch == epoch]
                if s.grp.nunique() < 2:
                    continue
                mdf = _fit(s, FORMULA5)
                beta, se, p0, vsref, omni = _interaction_betas(mdf, 'is_pref', GROUPS5, 'exc')
                for g in GROUPS5:
                    if g in beta:
                        ctrl.append(dict(theta=th, epoch=epoch, rep=rep, grp=g,
                                         beta=beta[g], omnibus_p=omni))
        print(f'  theta={th:.2f} done ({time.time()-t0:.0f}s elapsed)')
    ctrl_df = pd.DataFrame(ctrl)
    print(f'\nmatched-n control: {len(ctrl_df)} rows, {N_DROP_REPEATS} repeats per threshold')
else:
    print('matched-n control skipped (RANDOM_DROP_CONTROL = False)')

In [ ]:
# [3.7] titration plot: beta vs theta per group (matched-n band shaded) + omnibus p vs theta
fig, axes = plt.subplots(2, len(EPOCHS), figsize=(5.6 * len(EPOCHS), 7.4), squeeze=False)
for ci, epoch in enumerate(EPOCHS):
    ax = axes[0][ci]
    te = tit[tit.epoch == epoch]
    for g in GROUPS5:
        b = te[te.grp == g].sort_values('theta')
        if not len(b):
            continue
        ax.errorbar(b.theta, b.beta, yerr=b.se, color=GC5[g], marker='o', ms=5, lw=1.5,
                    capsize=2, label=g)
        if len(ctrl_df):                       # matched-n band: what any subset of that size gives
            c = ctrl_df[(ctrl_df.epoch == epoch) & (ctrl_df.grp == g)]
            if len(c):
                q = c.groupby('theta').beta.quantile([0.025, 0.975]).unstack()
                ax.fill_between(q.index, q[0.025], q[0.975], color=GC5[g], alpha=0.12, lw=0)
    ax.axhline(0, color='k', lw=0.7)
    ax.set_xlabel('subtype-confidence threshold θ'); ax.set_ylabel('is_pref β ± 1 SE')
    ax.set_title(f'{epoch}  —  β vs θ  (shaded = matched-n random drop, 95%)', fontsize=9)
    if ci == 0:
        ax.legend(fontsize=7, ncol=5, loc='upper center', bbox_to_anchor=(0.5, -0.16))

    ax2 = axes[1][ci]
    o = tit_omni[tit_omni.epoch == epoch].sort_values('theta')
    ax2.plot(o.theta, o.omnibus_p, 'o-', color='k', lw=1.5, ms=5, label='observed')
    if len(ctrl_df):
        c = ctrl_df[ctrl_df.epoch == epoch].drop_duplicates(['theta', 'rep'])
        if len(c):
            q = c.groupby('theta').omnibus_p.quantile([0.025, 0.975]).unstack()
            ax2.fill_between(q.index, q[0.025], q[0.975], color='k', alpha=0.10, lw=0,
                             label='matched-n 95%')
    ax2.axhline(ALPHA, color='r', ls=':', lw=1, label=f'p={ALPHA}')
    ax2.set_yscale('log'); ax2.set_xlabel('subtype-confidence threshold θ')
    ax2.set_ylabel('omnibus p'); ax2.set_title(f'{epoch}  —  omnibus p vs θ', fontsize=9)
    for _, r_ in o.iterrows():
        ax2.annotate(f'n={int(r_.n_inh)}', (r_.theta, r_.omnibus_p), textcoords='offset points',
                     xytext=(0, 7), ha='center', fontsize=7)
    ax2.legend(fontsize=7)
fig.suptitle('Section 3 across subtype-confidence thresholds (exc never thresholded)', fontsize=11)
fig.tight_layout(); plt.show()

## Section 4 — Save

In [ ]:
# [4.1] save betas + provenance
OUT = CT
beta_df.to_csv(OUT / f'lme_selectivity_betas_{region_tag}.csv', index=False)
bt5.to_csv(OUT / f'lme_selectivity_betas_subtypes_{region_tag}.csv', index=False)
tit.to_csv(OUT / f'lme_selectivity_conf_titration_{region_tag}.csv', index=False)
tit_omni.to_csv(OUT / f'lme_selectivity_conf_titration_omnibus_{region_tag}.csv', index=False)
if len(ctrl_df):
    ctrl_df.to_csv(OUT / f'lme_selectivity_conf_titration_control_{region_tag}.csv', index=False)
coef = pd.concat([pd.DataFrame(dict(epoch=e, term=m.params.index, beta=m.params.values,
                                    se=m.bse.values, z=m.tvalues.values, p=m.pvalues.values))
                  for e, m in fits.items()], ignore_index=True)
coef.to_csv(OUT / f'lme_selectivity_coefs_{region_tag}.csv', index=False)

prov = dict(dataset=DATASET, areas=SELECT_AREAS, load=LOAD, label='costas_class',
            model=FORMULA, fit='one LME per epoch',
            random_effects="groups=subject (intercept); vc_formula={'unit': '0 + C(unit_id)'} "
                           "(unit intercept). Random intercepts only, no slopes.",
            response='per-trial rate, baseline z-scored per cell',
            min_class_conf=MIN_CLASS_CONF,
            baseline=list(BASELINE), sd_floor=SD_FLOOR,
            enc_win=list(ENC_WIN), delay_win=list(DELAY_WIN), ref_class=REF_CLASS,
            n_rows=int(len(long_df)), n_cells=int(long_df.unit_id.nunique()),
            n_by_class=long_df.groupby('celltype').unit_id.nunique().to_dict(),
            converged={e: bool(m.converged) for e, m in fits.items()},
            betas={f'{r.epoch}_{r.celltype}': float(r.beta) for r in beta_df.itertuples()},
            p_vs0={f'{r.epoch}_{r.celltype}': float(r.p_vs0) for r in beta_df.itertuples()},
            p_class_diff={e: float(diff_p[e]) for e in diff_p},
            subtypes=dict(model=FORMULA5, groups=GROUPS5, exploratory=True,
                          n_cells={g: int(tab5.n_cells.get(g, 0)) for g in GROUPS5},
                          betas={f'{r.epoch}_{r.grp}': float(r.beta) for r in bt5.itertuples()},
                          p_vs0={f'{r.epoch}_{r.grp}': float(r.p_vs0) for r in bt5.itertuples()},
                          p_omnibus={e: float(omni5[e]) for e in omni5},
                          p_vs_exc={f'{e}_{g}': float(vsexc5[(e, g)])
                                    for e in EPOCHS for g in INH_SUBTYPES
                                    if (e, g) in vsexc5}))
(OUT / f'lme_selectivity_{region_tag}.json').write_text(json.dumps(prov, indent=2, default=float))

print('=' * 76)
print(f'CATEGORY SELECTIVITY BY COSTAS CLASS (LME per epoch) — {DATASET} {region_tag}')
print(f'load-{LOAD} | {long_df.unit_id.nunique()} cells | {len(long_df):,} trial-rows')
print(f'{FORMULA}')
print("random: groups=subject (intercept) + vc unit intercept — no slopes")
print('=' * 76)
for e in EPOCHS:
    for g in GROUPS:
        r_ = beta_df[(beta_df.epoch == e) & (beta_df.celltype == g)]
        if len(r_):
            r_ = r_.iloc[0]
            print(f'  {e:9s} {g:3s}  beta = {r_.beta:+.4f} ± {r_.se:.4f}   (vs 0: p={r_.p_vs0:.4g})')
    print(f'  {e:9s} inh vs exc  interaction p = {diff_p.get(e, float("nan")):.4g}')
print('=' * 76)
print('wrote lme_selectivity_betas / _coefs CSVs + JSON')
beta_df

## Reading the result

- **`[2.1]`** — one LME per epoch. Both class βs come from the same treatment-coded fit, so within an
  epoch they are jointly estimated and directly comparable, and the `is_pref × celltype` interaction
  *is* the inh−exc difference with its p. The inh β and SE use the delta method, so they do not
  depend on which class is the reference.
- **`[2.2]`** — β ± 1 SE per class per epoch. Stars are **per-point** tests that a class's β differs
  from 0; the annotated p above each epoch is the **between-class** interaction. Those are different
  questions — a class can be significantly selective while the two classes do not differ.
- **`[2.3]`** — descriptive. Traces are z-scored at **matched 50 ms resolution** so their height is
  honest; the LME deliberately keeps the **window-scale** baseline SD, since its response is a
  sustained-window rate. Shaded spans: fixation, enc1 on screen (0–2.016 s), maintenance
  (2.016–4.701 s).
- **`[3.2]`** — the subtype breakdown. The **omnibus** is the joint test that all four subtypes match
  exc; the per-subtype p's are the individual contrasts against exc from the same fit.

**Interpretation limits.** Encoding is circular (selection used encoding + probe); the delay was held
out and is clean. Read the class contrast within an epoch. The two epochs are fit as separate models,
so their βs are not on a common inferential footing — a term being significant in one epoch and not
the other is not by itself evidence that the two epochs differ.

Prevalence is settled elsewhere and not re-tested here: 193 exc / 78 inh against 184 / 87 expected
from the base rate, p = 0.16 — equally many cells of each class are category cells, so the open
question is strength.